# 1. Өгөгдөл бэлтгэх: WR-ийн Combine хөдөлгөөн

**Зорилго:** Bayesian circular mixed-effects загварт орох эргэлтийн өнцгийн өгөгдлийг бэлтгэх.

Эхний шатанд `players.csv`, `combine_results.csv`, `combine_tracking.csv` ашиглана. Том `game_tracking_*.csv`, `player_play.csv` болон карьерын үр дүнг энэ шатанд уншихгүй. Тэдгээрийг дараа нь NFL-ийн үндсэн улирлын гүйцэтгэлтэй холбож, баталгаажуулахад хэрэглэнэ.

Энэ хэсэг загвар сургахгүй, тоглогчдыг сайн/муу гэж эрэмбэлэхгүй. Нийт tracking-д BALL мөр ордог тул PLAYER мөрийг л сонгоно. Өнцгийг дараалсан гурван координатаас гаргана; өөр сорил, оролдлого болон хугацааны тасалдлыг хооронд нь холбохгүй.

Аргын суурь: Nguyen & Yurko (2025), *A Bayesian circular mixed-effects model for explaining variability in directional movement in American football*, https://arxiv.org/abs/2507.06122v1. Эндэх код нь өгүүллийн бүтэн загварыг давтсан хэрэгжилт биш; WR Combine өгөгдөлд зориулсан бэлтгэл юм.


In [ ]:
# 1A. Файлуудыг унших, бүтэц шалгах, WR тоглогчдыг сонгох
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 2027
rng = np.random.default_rng(SEED)
# Colab-д файлууд /content дотор байна. Өөр газар бол NFL_DATA_DIR-ийг тохируулна.
DATA_DIR = Path(os.environ.get("NFL_DATA_DIR", "/content"))
OUTPUT_DIR = DATA_DIR / "wr_preprocessing_outputs"
REQUIRED = {
    "players": ("players.csv", ["nfl_id", "display_name", "draft_year", "nfl_position"]),
    "combine_results": ("combine_results.csv", ["nfl_id", "draft_year", "forty"]),
    "combine_tracking": ("combine_tracking.csv", [
        "draft_year", "event_id", "nfl_id", "entity_type", "time", "drill_type",
        "drill_name", "attempt", "x", "y", "s", "a", "dis", "dir"
    ]),
}
missing_files = [filename for filename, _ in REQUIRED.values()
                 if not (DATA_DIR / filename).is_file()]
if missing_files:
    raise FileNotFoundError(f"{DATA_DIR} дотор файл дутуу: {missing_files}")

frames = {}
for name, (filename, required_columns) in REQUIRED.items():
    df = pd.read_csv(DATA_DIR / filename,
                     dtype={"event_id": "string"} if name == "combine_tracking" else None)
    absent = set(required_columns) - set(df.columns)
    if absent:
        raise ValueError(f"{filename}: багана дутуу: {sorted(absent)}")
    frames[name] = df
players = frames["players"]
combine_results = frames["combine_results"]
combine_tracking = frames["combine_tracking"]

for name, df in [("players", players), ("combine_results", combine_results)]:
    if df[["nfl_id", "draft_year"]].isna().any().any():
        raise ValueError(f"{name}: тоглогч/жилийн түлхүүрт дутуу утга байна")
    if df["nfl_id"].duplicated().any():
        raise ValueError(f"{name}: nfl_id давхардсан; холболтыг эхлээд засна уу")

players["nfl_position"] = players["nfl_position"].astype("string").str.strip().str.upper()
wr_players = players.loc[players["nfl_position"].eq("WR")].copy()
if wr_players.empty:
    raise ValueError("WR тоглогч олдсонгүй. Байрлалын кодыг шалгана уу.")
wr_ids = set(wr_players["nfl_id"])
wr_combine_results = combine_results.loc[combine_results["nfl_id"].isin(wr_ids)].copy()
wr_tracking = combine_tracking.loc[
    combine_tracking["entity_type"].astype("string").str.upper().eq("PLAYER")
    & combine_tracking["nfl_id"].isin(wr_ids)
].copy()
wr_tracking = wr_tracking.merge(
    wr_players[["nfl_id", "display_name", "draft_year"]].rename(
        columns={"draft_year": "player_draft_year"}),
    on="nfl_id", how="left", validate="many_to_one"
)
if not wr_tracking["draft_year"].eq(wr_tracking["player_draft_year"]).all():
    raise ValueError("Combine-ийн жил тоглогчийн draft_year-тай зөрж байна")

file_summary = pd.DataFrame([
    {"table": name, "rows": len(df), "columns": df.shape[1],
     "memory_MB": round(df.memory_usage(deep=True).sum() / 1024**2, 2)}
    for name, df in frames.items()
])
print("Уншсан файлууд:")
display(file_summary)
print(f"WR: {len(wr_players)} тоглогч | PLAYER tracking: {len(wr_tracking):,} мөр")
print("Байрлалын тархалт:")
display(players["nfl_position"].value_counts().rename_axis("position").to_frame("players"))
print("WR сорилуудын хамрах хүрээ:")
display(wr_tracking.groupby("drill_type").agg(
    players=("nfl_id", "nunique"), events=("event_id", "nunique"),
    frames=("nfl_id", "size")
).sort_values("players", ascending=False))
print("WR tracking-ийн дутуу утга (%):")
display(wr_tracking[REQUIRED["combine_tracking"][1]].isna().mean().mul(100)
        .round(2).to_frame("missing_pct"))
print("Тайлбар: frame-ийн тоо нь бие даасан тоглогчийн түүврийн хэмжээ биш.")


In [ ]:
# 1B. Дараалсан координатаас эргэлтийн өнцөг гаргах
# Өгөгдсөн s = хурд. step_length_yards = frame хоорондын шилжилт; эдгээр өөр хэмжигдэхүүн.
EXPECTED_DT_S = 0.10
DT_TOLERANCE_S = 0.015
MIN_SPEED_YD_S = 1.0  # Эхний QC босго; дараа нь 0.5/1.0/1.5 дээр мэдрэмжийг шалгана.
MAX_SPEED_YD_S = 15.0  # Зөвхөн илт зөрүүтэй координатыг QC-д тэмдэглэх өргөн босго.
SEQ = ["nfl_id", "draft_year", "event_id", "drill_type", "drill_name", "attempt"]
KEY = SEQ + ["timestamp"]

wr_motion = wr_tracking.copy()
wr_motion["timestamp"] = pd.to_datetime(wr_motion["time"], errors="coerce", utc=True)
for column in ["x", "y", "s", "a", "dis", "dir"]:
    wr_motion[column] = pd.to_numeric(wr_motion[column], errors="coerce")
if wr_motion[SEQ + ["timestamp"]].isna().any().any():
    raise ValueError("Дарааллын түлхүүр эсвэл timestamp дутуу/буруу; эх өгөгдлийг шалгана уу")

# Ижил бичлэгийг л арилгана; ижил цагт зөрүүтэй байрлалтай бол чимээгүй сонгохгүй.
raw_rows = len(wr_motion)
wr_motion = wr_motion.drop_duplicates().copy()
if wr_motion.duplicated(KEY, keep=False).any():
    raise ValueError("Ижил тоглогч/сорил/оролдлого/цаг дээр зөрүүтэй бичлэг байна")
wr_motion = wr_motion.sort_values(KEY).reset_index(drop=True)
grouped = wr_motion.groupby(SEQ, sort=False, observed=True)
prev_x, prev_y = grouped["x"].shift(1), grouped["y"].shift(1)
next_x, next_y = grouped["x"].shift(-1), grouped["y"].shift(-1)
prev_time = grouped["timestamp"].shift(1)
next_time = grouped["timestamp"].shift(-1)
wr_motion["dt_prev_s"] = (wr_motion["timestamp"] - prev_time).dt.total_seconds()
wr_motion["dt_next_s"] = (next_time - wr_motion["timestamp"]).dt.total_seconds()

# v_in = p(t)-p(t-1), v_out = p(t+1)-p(t).
in_x, in_y = wr_motion["x"] - prev_x, wr_motion["y"] - prev_y
out_x, out_y = next_x - wr_motion["x"], next_y - wr_motion["y"]
wr_motion["step_length_yards"] = np.hypot(out_x, out_y)
wr_motion["speed_in_yd_s"] = np.hypot(in_x, in_y) / wr_motion["dt_prev_s"]
wr_motion["speed_out_yd_s"] = wr_motion["step_length_yards"] / wr_motion["dt_next_s"]
# atan2(cross, dot) нь -pi..pi завсарт тойргийн заагийг зөв тооцно.
cross = in_x * out_y - in_y * out_x
dot = in_x * out_x + in_y * out_y
wr_motion["turn_angle_raw_rad"] = np.arctan2(cross, dot)
wr_motion["qc_contiguous"] = (
    wr_motion["dt_prev_s"].sub(EXPECTED_DT_S).abs().le(DT_TOLERANCE_S)
    & wr_motion["dt_next_s"].sub(EXPECTED_DT_S).abs().le(DT_TOLERANCE_S)
)
wr_motion["qc_speed"] = (
    wr_motion["speed_in_yd_s"].between(MIN_SPEED_YD_S, MAX_SPEED_YD_S)
    & wr_motion["speed_out_yd_s"].between(MIN_SPEED_YD_S, MAX_SPEED_YD_S)
)
wr_motion["valid_turn"] = wr_motion["qc_contiguous"] & wr_motion["qc_speed"]
wr_motion["turn_angle_rad"] = wr_motion["turn_angle_raw_rad"].where(wr_motion["valid_turn"])
wr_motion["turn_angle_deg"] = np.degrees(wr_motion["turn_angle_rad"])
# Эдгээр нь сонгосон шалгуураар хүчинтэй өнцгүүд; бүх мөр wr_motion-д хэвээр.
wr_turns_all_drills = wr_motion.loc[wr_motion["valid_turn"]].copy()
# Эхний судалгааны түүвэр: WR байрлалын сорил. 40-yard/shuttle/three-cone-ийг тусад нь хадгална.
wr_skill_turns = wr_turns_all_drills.loc[
    wr_turns_all_drills["drill_type"].eq("SKILL_DRILLS_WR")
].copy()
if wr_skill_turns.empty:
    raise ValueError("QC-ийн дараа WR skill drill-ийн хүчинтэй өнцөг үлдсэнгүй")

qc_summary = pd.Series({
    "raw_WR_PLAYER_rows": raw_rows,
    "exact_duplicate_rows_removed": raw_rows - len(wr_motion),
    "rows_after_deduplication": len(wr_motion),
    "contiguous_three_point_rows": int(wr_motion["qc_contiguous"].sum()),
    "valid_turn_rows_all_drills": len(wr_turns_all_drills),
    "valid_turn_rows_WR_skill_drills": len(wr_skill_turns),
    "WR_players_with_valid_skill_turns": wr_skill_turns["nfl_id"].nunique(),
})
display(qc_summary.to_frame("count"))
print("QC-ийн дараах WR skill сорилууд:")
skill_coverage = wr_skill_turns.groupby("drill_name").agg(
    players=("nfl_id", "nunique"), events=("event_id", "nunique"),
    valid_turns=("turn_angle_rad", "size")
).sort_values("players", ascending=False)
display(skill_coverage)
print("Босго нь судалгааны эцсийн сонголт биш. Удаан хөдөлгөөний шуугиан ба тасалдлыг тэмдэглэв.")
print("Том өнцөг / их хэлбэлзэл = сайн гүйцэтгэл гэж одоогоор дүгнэхгүй.")


In [ ]:
# 1C. Харааны шалгалт, дараагийн шатанд хэрэглэх хүснэгтүүдийг хадгалах
# Хамгийн олон тоглогчтой нэг ижил нэртэй сорилыг л эхний зурагт харуулна.
example_drill = skill_coverage.index[0]
example_turns = wr_skill_turns.loc[wr_skill_turns["drill_name"].eq(example_drill)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(example_turns["turn_angle_deg"], bins=60, color="#2864a0")
axes[0].set(xlabel="Turn angle (degrees)", ylabel="Frames", title=example_drill)
plot_sample = example_turns.sample(min(5000, len(example_turns)), random_state=SEED)
axes[1].scatter(plot_sample["speed_in_yd_s"], plot_sample["turn_angle_deg"],
                s=7, alpha=0.2, color="#2864a0")
axes[1].set(xlabel="Incoming speed (yards/second)", ylabel="Turn angle (degrees)",
            title="QC only: speed and turning")
plt.tight_layout()
plt.show()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
wr_players.to_csv(OUTPUT_DIR / "wr_players.csv", index=False)
wr_combine_results.to_csv(OUTPUT_DIR / "wr_combine_results.csv", index=False)
wr_motion.to_csv(OUTPUT_DIR / "wr_motion_qc.csv", index=False)
wr_skill_turns.to_csv(OUTPUT_DIR / "wr_skill_turns.csv", index=False)
skill_coverage.to_csv(OUTPUT_DIR / "wr_skill_coverage.csv")
qc_summary.to_csv(OUTPUT_DIR / "qc_summary.csv", header=["count"])
fig.savefig(OUTPUT_DIR / "wr_turn_qc.png", dpi=160, bbox_inches="tight")
print(f"Хадгалсан: {OUTPUT_DIR}")
print("Дараагийн шат: сорилын хэсгийг ялгах; хурд, сорилын хэлбэрийг тооцсон circular mixed-effects загвар.")
print("Анхаарах: Colab runtime устахад /content-ийн файлууд устаж болно. Notebook-ийн код Drive-д хадгалагдана.")


In [ ]:
# 2B. Сорилын хэсгийг бусад тоглогчийн хөдөлгөөний загвараар ойролцоолох
# 2025 оны cohort-ыг энэ сургалтаас тусгаарлаж хадгална.
# phase нь cone/event-ийн жинхэнэ шошго биш, normalized path progress-ийн template zone.
from scipy.ndimage import gaussian_filter1d
import json
MODEL_DIR = OUTPUT_DIR / "bayesian_stage2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRILLS = ["DAGGER_ROUTE_LEFT", "SLANT_ROUTE_LEFT", "CURL_ROUTE_RIGHT"]
N_BINS = 20
FRAME_STRIDE = 4  # Ойролцоогоор 0.4 сек; хөрш 3-цэгийн өнцгүүдийн давхцлыг багасгана.
SEGMENT_KEYS = SEQ
motion2 = wr_motion.loc[wr_motion["drill_name"].isin(DRILLS)].copy()
motion2 = motion2.sort_values(SEGMENT_KEYS + ["timestamp"]).reset_index(drop=True)
# Хугацааны тасалдал бүрд шинэ segment эхлүүлнэ.
new_segment = ~motion2["dt_prev_s"].sub(EXPECTED_DT_S).abs().le(DT_TOLERANCE_S)
motion2["segment"] = new_segment.groupby([motion2[k] for k in SEGMENT_KEYS]).cumsum().astype(int)
SEG = SEGMENT_KEYS + ["segment"]
g2 = motion2.groupby(SEG, sort=False, observed=True)
motion2["frame_index"] = g2.cumcount()
motion2["prev_turn_rad"] = g2["turn_angle_rad"].shift(1)
motion2["prev_speed_yd_s"] = g2["speed_in_yd_s"].shift(1)
motion2["signed_accel_yd_s2"] = (motion2["speed_in_yd_s"] - motion2["prev_speed_yd_s"]) / motion2["dt_prev_s"]
# Өмнөх цэгээс ирсэн алхмыг хурд * хугацаагаар сэргээнэ.
incoming_distance = motion2["speed_in_yd_s"] * motion2["dt_prev_s"]
incoming_distance = incoming_distance.where(~new_segment, 0).fillna(0)
motion2["path_distance"] = incoming_distance.groupby([motion2[k] for k in SEG]).cumsum()
path_total = motion2.groupby(SEG)["path_distance"].transform("max")
motion2["progress"] = motion2["path_distance"] / path_total.replace(0, np.nan)
motion2["progress_bin"] = np.minimum((motion2["progress"].fillna(0) * N_BINS).astype(int), N_BINS - 1)

candidate = motion2.loc[motion2["valid_turn"] & np.isfinite(motion2["prev_turn_rad"])
    & np.isfinite(motion2["signed_accel_yd_s2"]) & motion2["progress"].notna()].copy()
train_pool = candidate.loc[candidate["draft_year"].isin([2023, 2024])].copy()
if train_pool["nfl_id"].nunique() < 20:
    raise ValueError("Хөгжүүлэлтийн түүвэр хэт бага байна")
# Тоглогч бүр ижил жинтэй: олон frame/оролдлоготой хүн template-ийг эзлэхгүй.
per_player_bin = train_pool.groupby(["drill_name", "nfl_id", "progress_bin"])["turn_angle_rad"].apply(lambda s: np.abs(s).mean())
template_cache = {}
def make_template(drill, excluded_player=None):
    key = (drill, excluded_player)
    if key not in template_cache:
        tab = per_player_bin.loc[drill].unstack("progress_bin")
        if excluded_player is not None:
            tab = tab.drop(index=excluded_player, errors="ignore")
        curve = tab.mean(axis=0).reindex(range(N_BINS)).interpolate(limit_direction="both")
        if curve.isna().any():
            raise ValueError(f"{drill}: template үүсгэх боломжгүй")
        curve = gaussian_filter1d(curve.to_numpy(), sigma=1.0, mode="nearest")
        lower, upper = np.quantile(curve, [0.50, 0.75])
        zones = np.where(curve >= upper, "turn_zone", np.where(curve <= lower, "straight_zone", "transition_zone"))
        template_cache[key] = (curve, zones)
    return template_cache[key]

candidate["phase"] = ""
for (drill, player_id), idx in candidate.groupby(["drill_name", "nfl_id"]).groups.items():
    year = int(candidate.loc[idx, "draft_year"].iloc[0])
    # Сургалтын хүний phase-ийг өөрийн өнцгийг ашиглахгүйгээр тодорхойлно.
    _, zones = make_template(drill, player_id if year in [2023, 2024] else None)
    candidate.loc[idx, "phase"] = zones[candidate.loc[idx, "progress_bin"].to_numpy()]

model_data = candidate.loc[candidate["draft_year"].isin([2023, 2024]) & candidate["frame_index"].mod(FRAME_STRIDE).eq(0)].copy()
holdout_2025 = candidate.loc[candidate["draft_year"].eq(2025) & candidate["frame_index"].mod(FRAME_STRIDE).eq(0)].copy()
# Нэг тоглогчид хангалттай frame байгаа эсэхийг харуулна; бага түүврийг нууцаар эрэмбэлэхгүй.
player_counts = model_data.groupby("nfl_id").size().rename("model_frames")
print("Сургалт: 2023–2024; тусгаарласан cohort: 2025")
print("Train:", len(model_data), "frames;", model_data.nfl_id.nunique(), "players")
print("Held-out 2025:", len(holdout_2025), "frames;", holdout_2025.nfl_id.nunique(), "players")
display(model_data.groupby(["drill_name", "phase"]).agg(players=("nfl_id", "nunique"), frames=("nfl_id", "size")))
phase_template = pd.concat([pd.DataFrame({"drill_name": d, "progress": (np.arange(N_BINS)+0.5)/N_BINS,
    "mean_abs_turn_rad": make_template(d)[0], "phase": make_template(d)[1]}) for d in DRILLS], ignore_index=True)
fig2, axs2 = plt.subplots(1, len(DRILLS), figsize=(15, 4), sharey=True)
for ax, drill in zip(axs2, DRILLS):
    t = phase_template.loc[phase_template.drill_name.eq(drill)]
    ax.plot(t.progress, np.degrees(t.mean_abs_turn_rad), color="gray")
    for phase, color in [("straight_zone", "#2864a0"), ("transition_zone", "#999999"), ("turn_zone", "#d97706")]:
        s = t.loc[t.phase.eq(phase)]
        ax.scatter(s.progress, np.degrees(s.mean_abs_turn_rad), color=color, label=phase)
    ax.set(title=drill, xlabel="Fraction of path distance")
axs2[0].set_ylabel("Template mean absolute turn (degrees)")
axs2[-1].legend(fontsize=7)
plt.tight_layout(); plt.show()
fig2.savefig(MODEL_DIR / "phase_templates.png", dpi=150, bbox_inches="tight")
model_data.to_csv(MODEL_DIR / "model_data_2023_2024.csv", index=False)
holdout_2025.to_csv(MODEL_DIR / "heldout_2025_not_fitted.csv", index=False)
phase_template.to_csv(MODEL_DIR / "phase_templates.csv", index=False)
print("Анхаарах: zones нь бусад тоглогчийн template дээрх харьцангуй их/бага эргэлтийн хэсэг; баталгаатай cone шошго биш.")


In [ ]:
# 2C. Bayesian circular mixed-effects загвар: mean болон concentration хоёрыг загварчилна.
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from pymc.distributions.dist_math import i0e
print("PyMC", pm.__version__, "ArviZ", az.__version__)

# Continuous covariates-ийг зөвхөн сургалтын өгөгдлөөр стандартчилна.
CONT = ["speed_in_yd_s", "signed_accel_yd_s2", "progress"]
centers = model_data[CONT].mean()
scales = model_data[CONT].std().replace(0, 1)
Z = (model_data[CONT] - centers) / scales
# Байрлалын ганц бүлэг WR тул position variance байхгүй; player random effect байна.
CONTEXT = pd.get_dummies(model_data[["drill_name", "phase"]], drop_first=True, dtype=float)
X_mu = np.column_stack([np.ones(len(model_data)), Z.to_numpy(),
    model_data["prev_turn_rad"].to_numpy(), CONTEXT.to_numpy()])
X_kappa = np.column_stack([np.ones(len(model_data)), Z.to_numpy(), CONTEXT.to_numpy()])
mu_names = ["intercept"] + CONT + ["previous_turn_rad"] + CONTEXT.columns.tolist()
kappa_names = ["intercept"] + CONT + CONTEXT.columns.tolist()
player_ids = np.sort(model_data["nfl_id"].unique())
player_idx = pd.Categorical(model_data["nfl_id"], categories=player_ids).codes
# Оролдлогын random intercept нь нэг оролдлогын үлдэгдэл холбоог хэсэгчлэн тооцно.
attempt_labels = model_data[SEQ].astype(str).agg("|".join, axis=1)
attempt_idx, attempt_levels = pd.factorize(attempt_labels, sort=True)
y_angle = model_data["turn_angle_rad"].to_numpy()
coords = {"player": player_ids.astype(str), "attempt": attempt_levels.tolist(),
    "mu_feature": mu_names, "kappa_feature": kappa_names}
with pm.Model(coords=coords) as circular_model:
    beta_mu = pm.Normal("beta_mu", mu=0, sigma=0.3, dims="mu_feature")
    gamma_prior = np.zeros(X_kappa.shape[1]); gamma_prior[0] = np.log(400)
    gamma_scale = np.full(X_kappa.shape[1], 0.7); gamma_scale[0] = 1.5
    beta_kappa = pm.Normal("beta_kappa", mu=gamma_prior, sigma=gamma_scale, dims="kappa_feature")
    sigma_player = pm.HalfNormal("sigma_player", sigma=0.7)
    player_z = pm.Normal("player_z", 0, 1, dims="player")
    player_effect = pm.Deterministic("player_effect", sigma_player * player_z, dims="player")
    sigma_attempt = pm.HalfNormal("sigma_attempt", sigma=0.7)
    attempt_z = pm.Normal("attempt_z", 0, 1, dims="attempt")
    eta_mu = pt.dot(X_mu, beta_mu)
    mu_angle = 2 * pt.arctan(eta_mu)
    log_kappa = pt.dot(X_kappa, beta_kappa) + player_effect[player_idx] + sigma_attempt * attempt_z[attempt_idx]
    # PyMC 5.28-ийн log(I0(kappa)) overflow-оос сэргийлсэн яг ижил von Mises logp.
    # I0e(kappa) = exp(-kappa) I0(kappa), kappa > 0.
    def stable_vonmises_logp(value, mu, kappa):
        return kappa * (pt.cos(value - mu) - 1) - pt.log(2 * np.pi) - pt.log(i0e(kappa))
    pm.CustomDist("turn", mu_angle, pt.exp(log_kappa),
        logp=stable_vonmises_logp, observed=y_angle)
    initial_logp = circular_model.compile_logp()(circular_model.initial_point())
    if not np.isfinite(initial_logp):
        raise ValueError("Initial logp finite биш; sampling эхлүүлэхгүй")
    print("Finite initial logp:", float(initial_logp))
    idata = pm.sample(draws=750, tune=1000, chains=4, cores=2,
        target_accept=0.95, init="adapt_diag", random_seed=SEED,
        idata_kwargs={"log_likelihood": False})
# Нэн түрүүнд хадгална: оношилгоо хангахгүй байсан ч сургалтын үр дүн алдагдахгүй.
idata.to_netcdf(MODEL_DIR / "circular_model_posterior.nc")
model_spec = {"method": "Bayesian von Mises mean/concentration mixed-effects",
    "training_years": [2023, 2024], "heldout_year": 2025,
    "drills": DRILLS, "frame_stride": FRAME_STRIDE,
    "centers": centers.to_dict(), "scales": scales.to_dict(),
    "mu_features": mu_names, "kappa_features": kappa_names,
    "note": "Exploratory Combine trait model; not validated against NFL game performance."}
(MODEL_DIR / "model_spec.json").write_text(json.dumps(model_spec, ensure_ascii=False, indent=2))
print("Posterior хадгалсан:", MODEL_DIR)


In [ ]:
# 2D. Нийлэлт, posterior predictive шалгалт, тоглогчийн uncertainty
# Өнцгийн хэлбэлзэл нь тоглолтын сайн гүйцэтгэл гэсэн баталгаа биш.
VARS = ["beta_mu", "beta_kappa", "sigma_player", "sigma_attempt", "player_effect", "player_z", "attempt_z"]
diagnostics = az.summary(idata, var_names=VARS, hdi_prob=0.95)
display(diagnostics)
divergences = int(idata.sample_stats["diverging"].sum())
rhat_values = az.rhat(idata, var_names=VARS)
max_rhat = max(float(v.max()) for v in rhat_values.data_vars.values())
ess_values = az.ess(idata, var_names=VARS, method="bulk")
min_ess = min(float(v.min()) for v in ess_values.data_vars.values())
bfmi = np.asarray(az.bfmi(idata))
max_tree_depth = int(idata.sample_stats["tree_depth"].max())
SAMPLER_OK = (divergences == 0 and max_rhat <= 1.01 and min_ess >= 400 and bfmi.min() > 0.3 and max_tree_depth < 10)
print({"divergences": divergences, "max_rhat": max_rhat, "min_bulk_ESS": min_ess,
       "min_BFMI": float(bfmi.min()), "max_tree_depth": max_tree_depth, "sampler_checks_pass": SAMPLER_OK})
if not SAMPLER_OK:
    print("Оношилгоо хангаагүй: тоглогчийн дүгнэлт хийхээс өмнө tune/draws/загварыг дахин шалгана.")
diagnostics.to_csv(MODEL_DIR / "mcmc_diagnostics.csv")
# Random effect-ийн exp нь ижил нөхцөл дэх concentration multiplier.
pe = idata.posterior["player_effect"].transpose("chain", "draw", "player").values.reshape(-1, len(player_ids))
multiplier = np.exp(pe)
trait_table = pd.DataFrame({"nfl_id": player_ids, "concentration_multiplier_median": np.median(multiplier, axis=0),
    "ci95_low": np.quantile(multiplier, 0.025, axis=0), "ci95_high": np.quantile(multiplier, 0.975, axis=0)})
trait_table = trait_table.merge(wr_players[["nfl_id", "display_name", "draft_year"]], on="nfl_id", validate="one_to_one")
trait_table = trait_table.merge(player_counts, on="nfl_id", validate="one_to_one")
trait_table["sampler_checks_pass"] = SAMPLER_OK
trait_table["result_status"] = "EXPLORATORY_NOT_VALIDATED"
trait_table.to_csv(MODEL_DIR / "player_turn_traits_exploratory.csv", index=False)
print(">1: илүү төвлөрсөн, <1: илүү хэлбэлзэлтэй өнцөг; сайн/муу эрэмбэ биш.")
display(trait_table.sort_values("display_name").head(12))

# 120 posterior draws-оос one-step conditional predictive шалгалт.
# Энэ нь бүх замын симуляц болон NFL гүйцэтгэлийн шалгалт биш.
post = idata.posterior.stack(sample=("chain", "draw"))
prng = np.random.default_rng(SEED)
chosen = prng.choice(post.sizes["sample"], size=min(120, post.sizes["sample"]), replace=False)
bm = post["beta_mu"].transpose("sample", "mu_feature").values[chosen]
bk = post["beta_kappa"].transpose("sample", "kappa_feature").values[chosen]
u = post["player_effect"].transpose("sample", "player").values[chosen]
sa = post["sigma_attempt"].values[chosen]
azs = post["attempt_z"].transpose("sample", "attempt").values[chosen]
pp_mu = 2 * np.arctan(bm @ X_mu.T)
pp_kappa = np.exp(bk @ X_kappa.T + u[:, player_idx] + sa[:, None] * azs[:, attempt_idx])
y_rep = prng.vonmises(pp_mu, pp_kappa)
pp_checks = pd.DataFrame({"statistic": ["mean_cos", "mean_sin", "abs_angle_90pct"],
    "observed": [np.cos(y_angle).mean(), np.sin(y_angle).mean(), np.quantile(np.abs(y_angle), 0.9)],
    "replicated_2.5pct": [np.quantile(np.cos(y_rep).mean(axis=1), .025), np.quantile(np.sin(y_rep).mean(axis=1), .025), np.quantile(np.quantile(np.abs(y_rep), .9, axis=1), .025)],
    "replicated_97.5pct": [np.quantile(np.cos(y_rep).mean(axis=1), .975), np.quantile(np.sin(y_rep).mean(axis=1), .975), np.quantile(np.quantile(np.abs(y_rep), .9, axis=1), .975)]})
print("Conditional posterior predictive checks:"); display(pp_checks)
pp_checks["inside_predictive_interval"] = pp_checks["observed"].between(pp_checks["replicated_2.5pct"], pp_checks["replicated_97.5pct"])
pp_checks.to_csv(MODEL_DIR / "posterior_predictive_checks.csv", index=False)
PPC_OK = bool(pp_checks["inside_predictive_interval"].all())
trait_table["predictive_checks_pass"] = PPC_OK
trait_table.to_csv(MODEL_DIR / "player_turn_traits_exploratory.csv", index=False)
print("Global predictive checks pass:", PPC_OK)
if not PPC_OK:
    print("Загвар тархалтын зарим шинжийг давтаж чадахгүй байна; traits-ийг баталгаатай үнэлгээ гэж ашиглахгүй.")
# Нэг оролдлогын sampled residual-ууд холбоотой үлдсэн эсэх (descriptive diagnostic).
residual = np.angle(np.exp(1j * (y_angle - np.angle(np.exp(1j * pp_mu).mean(axis=0)))))
res_table = model_data[SEQ + ["timestamp"]].copy()
res_table["residual"] = residual
res_table = res_table.sort_values(SEQ + ["timestamp"])
res_table["previous_residual"] = res_table.groupby(SEQ)["residual"].shift(1)
residual_lag_corr = float(res_table[["residual", "previous_residual"]].corr().iloc[0, 1])
print("Within-attempt sampled residual lag correlation:", residual_lag_corr)
(MODEL_DIR / "diagnostic_status.json").write_text(json.dumps({"sampler_checks_pass": bool(SAMPLER_OK),
    "predictive_checks_pass": PPC_OK, "max_rhat": max_rhat, "min_bulk_ess": min_ess,
    "divergences": divergences, "residual_lag_correlation": residual_lag_corr}, indent=2))
fig3, axs3 = plt.subplots(1, 2, figsize=(13, 5))
# Gaussian-like tight angles-г ойлгомжтой харуулахын тулд degrees ашиглав; likelihood нь circular хэвээр.
bins = np.linspace(-30, 30, 70)
axs3[0].hist(np.degrees(y_angle), bins=bins, density=True, alpha=.5, label="Observed")
axs3[0].hist(np.degrees(y_rep).ravel(), bins=bins, density=True, histtype="step", color="orange", label="Posterior predictive")
axs3[0].set(xlabel="Turn angle (degrees)", title="Conditional fit (display range +/-30 deg)"); axs3[0].legend()
# Бүх хүнийг эрэмбэлэхгүй, нэрийн цагаан толгойн эхний 15-ыг харуулна.
show = trait_table.sort_values("display_name").head(15).iloc[::-1]
axs3[1].errorbar(show.concentration_multiplier_median, np.arange(len(show)),
    xerr=[show.concentration_multiplier_median-show.ci95_low, show.ci95_high-show.concentration_multiplier_median], fmt="o", capsize=2)
axs3[1].set_yticks(np.arange(len(show)), labels=show.display_name)
axs3[1].axvline(1, color="gray", linestyle="--")
axs3[1].set(xlabel="Concentration multiplier (95% credible interval)", title="Exploratory traits; alphabetical subset")
plt.tight_layout(); plt.show()
fig3.savefig(MODEL_DIR / "bayesian_diagnostics_traits.png", dpi=150, bbox_inches="tight")
print("2025 cohort-ыг сургалт, template байгуулахад ашиглаагүй. Held-out үнэлгээг дараагийн шатанд хийнэ.")
print("Convergence хангах нь загвар зөв эсвэл NFL-д ашигтайг дангаараа батлахгүй.")
print("Хадгалсан:", MODEL_DIR)


In [ ]:
# 3A. V2: сорил тус бүрийн явцыг spline-ээр, оролдлогын ялгааг mean random effect-ээр тооцох
# V1-ийг өөрчлөхгүй. 2025 өгөгдөл сургалт болон тохируулгад орохгүй.
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge
import joblib
V2_DIR = MODEL_DIR / "v2_spline"
V2_DIR.mkdir(parents=True, exist_ok=True)
idata_v1 = idata
X_mu_v1, X_kappa_v1 = X_mu.copy(), X_kappa.copy()

progress_spline = SplineTransformer(n_knots=4, degree=3, include_bias=False, extrapolation="constant")
B = progress_spline.fit_transform(model_data[["progress"]])
D = pd.get_dummies(model_data["drill_name"], dtype=float).reindex(columns=DRILLS)
interaction_blocks = [B * D[d].to_numpy()[:, None] for d in DRILLS]
mu2_names = ["intercept", "speed_z", "signed_accel_z", "previous_turn_rad"]
Xmu2 = np.column_stack([np.ones(len(model_data)), Z["speed_in_yd_s"], Z["signed_accel_yd_s2"],
    model_data["prev_turn_rad"], D.iloc[:, 1:].to_numpy(), *interaction_blocks])
mu2_names += ["drill_" + d for d in DRILLS[1:]]
mu2_names += [f"{d}_progress_basis_{j}" for d in DRILLS for j in range(B.shape[1])]
speed_sq = Z["speed_in_yd_s"].to_numpy() ** 2
speed_sq_center, speed_sq_scale = speed_sq.mean(), speed_sq.std()
speed_sq_z = (speed_sq - speed_sq_center) / speed_sq_scale
phase_dummies = pd.get_dummies(model_data["phase"], drop_first=True, dtype=float)
Xkap2 = np.column_stack([np.ones(len(model_data)), Z["speed_in_yd_s"], speed_sq_z,
    Z["signed_accel_yd_s2"], D.iloc[:, 1:].to_numpy(), phase_dummies.to_numpy()])
kap2_names = ["intercept", "speed_z", "speed_squared_z", "signed_accel_z"]
kap2_names += ["drill_" + d for d in DRILLS[1:]] + phase_dummies.columns.tolist()
# Prior биш: зөвхөн NUTS-ийн эхлэх цэгийг data-д ойр болгох ridge initialization.
initial_beta_mu = Ridge(alpha=2., fit_intercept=False).fit(Xmu2, np.tan(y_angle/2)).coef_
# Orthogonal basis: span нь ижил; prior-ыг QR coefficient дээр тогтоосон V2.
Xmu2_original = Xmu2.copy()
Q2, R2 = np.linalg.qr(Xmu2_original, mode="reduced")
qr_scale2 = np.sqrt(len(model_data))
Xmu2 = Q2 * qr_scale2
R2_scaled = R2 / qr_scale2
initial_beta_mu = R2_scaled @ initial_beta_mu
mu2_original_names = mu2_names.copy()
mu2_names = [f"qr_mean_{j}" for j in range(Xmu2.shape[1])]
np.save(V2_DIR / "mean_basis_R_scaled.npy", R2_scaled)
print("Mean design rank:", np.linalg.matrix_rank(Xmu2_original), "of", Xmu2_original.shape[1])
print("V1 dispersion hyperparameters:")
display(az.summary(idata_v1, var_names=["sigma_player", "sigma_attempt"], hdi_prob=.95))
print("V2 same training rows:", Xmu2.shape[0], "players:", len(player_ids))
print("Mean coefficients:", Xmu2.shape[1], "Concentration coefficients:", Xkap2.shape[1])
joblib.dump(progress_spline, V2_DIR / "progress_spline.joblib")


In [ ]:
# 3B. V2 Bayesian circular mixed-effects fit
# Mean нь drill-specific smooth curve + speed/acceleration/previous turn + attempt intercept.
# Concentration нь nonlinear speed + context + player random effect.
coords2 = {"player": player_ids.astype(str), "attempt": attempt_levels.tolist(),
           "mu_feature": mu2_names, "kappa_feature": kap2_names}
with pm.Model(coords=coords2) as circular_model_v2:
    beta_mu2 = pm.Normal("beta_mu", 0., .05, dims="mu_feature")
    k_prior2 = np.zeros(Xkap2.shape[1]); k_prior2[0] = np.log(400.)
    k_sd2 = np.full(Xkap2.shape[1], .7); k_sd2[0] = 1.5
    beta_kappa2 = pm.Normal("beta_kappa", k_prior2, k_sd2, dims="kappa_feature")
    sigma_player2 = pm.HalfNormal("sigma_player", .7)
    player_z2 = pm.Normal("player_z", 0., 1., dims="player")
    player_effect2 = pm.Deterministic("player_effect", sigma_player2 * player_z2, dims="player")
    sigma_attempt_mean2 = pm.HalfNormal("sigma_attempt_mean", .03)
    attempt_z2 = pm.Normal("attempt_z", 0., 1., dims="attempt")
    attempt_mean2 = sigma_attempt_mean2 * (attempt_z2[attempt_idx] - pt.mean(attempt_z2))
    mu2 = 2 * pt.arctan(pt.dot(Xmu2, beta_mu2) + attempt_mean2)
    kappa2 = pt.exp(pt.dot(Xkap2, beta_kappa2) + player_effect2[player_idx])
    pm.CustomDist("turn", mu2, kappa2, logp=stable_vonmises_logp, observed=y_angle)
    assert np.isfinite(circular_model_v2.compile_logp()(circular_model_v2.initial_point()))
    idata_v2 = pm.sample(draws=1200, tune=1000, chains=4, cores=2, target_accept=.95,
        init="adapt_diag", initvals={"beta_mu": initial_beta_mu}, random_seed=SEED+1,
        idata_kwargs={"log_likelihood": False})
idata_v2.to_netcdf(V2_DIR / "posterior.nc")
(V2_DIR / "model_spec.json").write_text(json.dumps({"training_years":[2023,2024], "heldout_year":2025,
    "mu_features":mu2_names, "original_mu_features":mu2_original_names, "mu_QR_prior_sd":.05, "kappa_features":kap2_names, "speed_sq_center":float(speed_sq_center),
    "speed_sq_scale":float(speed_sq_scale), "standardization_centers":centers.to_dict(),
    "standardization_scales":scales.to_dict(), "version":"v2_spline_attempt_mean"},indent=2))
print("V2 posterior хадгаллаа:", V2_DIR)


In [ ]:
# 3C. V1 ба V2-ийн ижил түүвэр дээрх оношилгооны харьцуулалт
vars2 = ["beta_mu", "beta_kappa", "sigma_player", "player_z", "player_effect", "sigma_attempt_mean", "attempt_z"]
diag2 = az.summary(idata_v2, var_names=vars2, hdi_prob=.95)
r2 = az.rhat(idata_v2, var_names=vars2)
e2 = az.ess(idata_v2, var_names=vars2, method="bulk")
max_rhat2 = max(float(v.max()) for v in r2.data_vars.values())
min_ess2 = min(float(v.min()) for v in e2.data_vars.values())
div2 = int(idata_v2.sample_stats.diverging.sum())
bfmi2 = float(np.min(az.bfmi(idata_v2)))
depth2 = int(idata_v2.sample_stats.tree_depth.max())
sampler_ok2 = bool(div2 == 0 and max_rhat2 <= 1.01 and min_ess2 >= 400 and bfmi2 > .3 and depth2 < 10)
diag2.to_csv(V2_DIR / "mcmc_diagnostics.csv")
p2 = idata_v2.posterior.stack(sample=("chain","draw"))
rng2 = np.random.default_rng(SEED+1)
sel2 = rng2.choice(p2.sizes["sample"], 600, replace=False)
bm2 = p2.beta_mu.transpose("sample","mu_feature").values[sel2]
bk2 = p2.beta_kappa.transpose("sample","kappa_feature").values[sel2]
u2 = p2.player_effect.transpose("sample","player").values[sel2]
zatt2 = p2.attempt_z.transpose("sample","attempt").values[sel2]
satt2 = p2.sigma_attempt_mean.values[sel2]
pmu2 = 2*np.arctan(bm2 @ Xmu2.T + satt2[:,None]*(zatt2[:,attempt_idx]-zatt2.mean(axis=1,keepdims=True)))
pkap2 = np.exp(bk2 @ Xkap2.T + u2[:,player_idx])
yrep2 = rng2.vonmises(pmu2, pkap2)
obs_stats = np.array([np.cos(y_angle).mean(), np.sin(y_angle).mean(), np.quantile(np.abs(y_angle),.9)])
rep_stats2 = np.column_stack([np.cos(yrep2).mean(axis=1),np.sin(yrep2).mean(axis=1),np.quantile(np.abs(yrep2),.9,axis=1)])
ppc2 = pd.DataFrame({"statistic":["mean_cos","mean_sin","abs_angle_90pct"], "observed":obs_stats,
    "predictive_low":np.quantile(rep_stats2,.025,axis=0),"predictive_high":np.quantile(rep_stats2,.975,axis=0)})
ppc2["inside_interval"] = ppc2.observed.between(ppc2.predictive_low,ppc2.predictive_high)
ppc_ok2 = bool(ppc2.inside_interval.all())
ppc2.to_csv(V2_DIR / "conditional_predictive_checks.csv",index=False)
resid2 = np.angle(np.exp(1j*(y_angle-np.angle(np.exp(1j*pmu2).mean(axis=0)))))
rt2 = model_data[SEQ+["timestamp"]].copy(); rt2["residual"]=resid2
rt2=rt2.sort_values(SEQ+["timestamp"])
rt2["lag"] = rt2.groupby(SEQ).residual.shift(1)
lagcorr2=float(rt2[["residual","lag"]].corr().iloc[0,1])
# Ижил 600 draws ашиглан V1 predictive шалгалтын Monte Carlo ялгааг багасгана.
p1=idata_v1.posterior.stack(sample=("chain","draw"))
s1=np.random.default_rng(SEED+1).choice(p1.sizes["sample"],600,replace=False)
m1=2*np.arctan(p1.beta_mu.transpose("sample","mu_feature").values[s1] @ X_mu_v1.T)
k1=np.exp(p1.beta_kappa.transpose("sample","kappa_feature").values[s1] @ X_kappa_v1.T
    + p1.player_effect.transpose("sample","player").values[s1][:,player_idx]
    + p1.sigma_attempt.values[s1,None]*p1.attempt_z.transpose("sample","attempt").values[s1][:,attempt_idx])
rep1=np.random.default_rng(SEED+1).vonmises(m1,k1)
rstat1=np.column_stack([np.cos(rep1).mean(axis=1),np.sin(rep1).mean(axis=1),np.quantile(np.abs(rep1),.9,axis=1)])
ppc1_pass=bool(np.all((obs_stats>=np.quantile(rstat1,.025,axis=0)) & (obs_stats<=np.quantile(rstat1,.975,axis=0))))
comparison=pd.DataFrame({"version":["V1","V2"],"divergences":[divergences,div2],"max_Rhat":[max_rhat,max_rhat2],
    "min_bulk_ESS":[min_ess,min_ess2],"min_BFMI":[float(bfmi.min()),bfmi2],"max_tree_depth":[max_tree_depth,depth2],
    "sampler_checks_pass":[SAMPLER_OK,sampler_ok2],"PPC_600_draws_pass":[ppc1_pass,ppc_ok2],
    "residual_lag_corr":[residual_lag_corr,lagcorr2]})
display(comparison); display(ppc2)
comparison.to_csv(V2_DIR / "v1_v2_comparison.csv",index=False)
upost=np.exp(idata_v2.posterior.player_effect.transpose("chain","draw","player").values.reshape(-1,len(player_ids)))
traits2=pd.DataFrame({"nfl_id":player_ids,"concentration_multiplier_median":np.median(upost,axis=0),
    "ci95_low":np.quantile(upost,.025,axis=0),"ci95_high":np.quantile(upost,.975,axis=0)})
traits2=traits2.merge(wr_players[["nfl_id","display_name","draft_year"]],on="nfl_id",validate="one_to_one").merge(player_counts,on="nfl_id")
traits2["sampler_checks_pass"]=sampler_ok2; traits2["predictive_checks_pass"]=ppc_ok2
traits2["result_status"]="EXPLORATORY_NOT_NFL_VALIDATED"
traits2.to_csv(V2_DIR / "player_traits_exploratory.csv",index=False)
print("<1: илүү хэлбэлзэлтэй, >1: илүү төвлөрсөн өнцөг; чадварын чансаа биш.")
figv2, axv2=plt.subplots(1,3,figsize=(15,4))
bins=np.linspace(-30,30,70)
for ax,rep,title in [(axv2[0],rep1,"V1 conditional fit"),(axv2[1],yrep2,"V2 conditional fit")]:
    ax.hist(np.degrees(y_angle),bins=bins,density=True,alpha=.4,label="Observed")
    ax.hist(np.degrees(rep).ravel(),bins=bins,density=True,histtype="step",color="orange",label="Predictive")
    ax.set(title=title,xlabel="Turn angle (degrees)"); ax.legend(fontsize=8)
show2=traits2.sort_values("display_name").head(12).iloc[::-1]
axv2[2].errorbar(show2.concentration_multiplier_median,np.arange(len(show2)),
    xerr=[show2.concentration_multiplier_median-show2.ci95_low,show2.ci95_high-show2.concentration_multiplier_median],fmt="o",capsize=2)
axv2[2].set_yticks(np.arange(len(show2)),labels=show2.display_name)
axv2[2].axvline(1,color="gray",linestyle="--")
axv2[2].set(title="V2 traits: alphabetical subset",xlabel="Concentration multiplier (95% CrI)")
plt.tight_layout(); plt.show()
figv2.savefig(V2_DIR / "v1_v2_diagnostics.png",dpi=150,bbox_inches="tight")
print("2025 болон NFL гүйцэтгэлийн өгөгдөл ашиглаагүй. Энэ бол conditional training-fit харьцуулалт.")
print("Нийлэлт сайн байсан ч predictive шалгалт болон residual холбоог тусад нь харна.")
print("Хадгалсан:",V2_DIR)


In [ ]:
# 3D. Үр дүнгийн шийдвэр ба runtime устахаас өмнөх нөөц
import shutil
from IPython.display import FileLink
v2_approved = bool(sampler_ok2 and ppc_ok2 and abs(lagcorr2) < 0.1)
traits2["result_status"] = "NOT_READY_FOR_PLAYER_COMPARISON" if not v2_approved else "EXPLORATORY_NOT_NFL_VALIDATED"
traits2.to_csv(V2_DIR / "player_traits_exploratory.csv", index=False)
findings = {
    "v2_sampler_checks_pass": sampler_ok2,
    "v2_predictive_checks_pass": ppc_ok2,
    "v2_max_Rhat": max_rhat2,
    "v2_min_bulk_ESS": min_ess2,
    "v2_divergences": div2,
    "v1_residual_lag_corr": residual_lag_corr,
    "v2_residual_lag_corr": lagcorr2,
    "v2_ready_for_player_comparison": v2_approved,
    "NFL_performance_validation_done": False,
    "heldout_2025_used_for_training": False,
}
(V2_DIR / "decision.json").write_text(json.dumps(findings, indent=2))
report_text = (
    "V2 MCMC convergence improved, but conditional predictive checks did not all pass.\n"
    f"Rhat={max_rhat2:.6f}; divergences={div2}; minimum bulk ESS={min_ess2:.1f}.\n"
    f"Sampled within-attempt residual lag correlation: V1={residual_lag_corr:.3f}, V2={lagcorr2:.3f}.\n"
    "Do not use these traits as validated player rankings.\n"
    "2025 cohort and NFL game outcomes were not used in training.\n"
    "Next: diagnose conditional directional bias and temporal dependence before NFL validation.\n"
)
(V2_DIR / "FINDINGS.txt").write_text(report_text)
print("V2: MCMC шалгалт тэнцсэн; predictive шалгалт бүрэн тэнцээгүй.")
print("Одоогийн төлөв:", traits2.result_status.iloc[0])
display(comparison.round(4))
display(figv2)
archive_path = Path(shutil.make_archive(str(DATA_DIR / "wr_bayesian_stage2_results"), "zip", root_dir=MODEL_DIR))
print("Нөөц ZIP бэлэн:", archive_path, f"({archive_path.stat().st_size / 1024**2:.1f} MiB)")
print("Энэ нөөцөд хоёр загварын posterior, CSV, оношилгоо, график, тайлбар багтсан.")


V2 executed in Colab: max Rhat 1.006988; divergences 0; minimum bulk ESS 653.44. Conditional predictive checks did not all pass. Residual lag correlation increased from 0.2662 to 0.3600. Not ready for player comparison. 2025 held out. Local backup contains source only; fitted posterior remains in Colab runtime/archive.

## V3: dynamic recorded-zero hurdle model
Development validation only: 2023 to 2024; 2025 untouched.

In [ ]:
# 5A. V3-ийг сонгохоос өмнөх шалтгаан оношилгоо. Зөвхөн 2023-2024.
V3_DIR = MODEL_DIR / 'v3_dynamic_hurdle'
V3_DIR.mkdir(parents=True, exist_ok=True)
audit3 = model_data.copy()
audit3['exact_zero'] = audit3.turn_angle_rad.eq(0)
audit3['near_zero_05deg'] = audit3.turn_angle_rad.abs().le(np.radians(.5))
audit3['obs_sin'] = np.sin(y_angle)
audit3['v2_pred_sin'] = np.sin(pmu2).mean(axis=0)
audit3['v2_residual'] = resid2
summary3 = audit3.groupby('drill_name').agg(frames=('nfl_id','size'),zero_rate=('exact_zero','mean'),near_zero_rate=('near_zero_05deg','mean'),obs_mean_sin=('obs_sin','mean'),pred_mean_sin=('v2_pred_sin','mean'))
display(summary3)
print('Zero rows:',int(audit3.exact_zero.sum()),'of',len(audit3))
print('Previous raw angle correlation:',np.corrcoef(y_angle,model_data.prev_turn_rad)[0,1])
print('Unique values near zero:',np.unique(np.round(np.degrees(y_angle[np.abs(y_angle)<.005]),8))[:25])
# Lag at four original frames; do not cross a time gap.
md3=motion2.sort_values(SEG+['timestamp']).copy()
g3=md3.groupby(SEG,sort=False)
md3['lag4_turn_rad']=g3.turn_angle_rad.shift(4)
md3['lag4_time']=g3.timestamp.shift(4)
md3['lag4_ok']=(md3.timestamp-md3.lag4_time).dt.total_seconds().sub(.4).abs().le(.025)
lag3=md3.set_index(KEY)[['lag4_turn_rad','lag4_ok']]
audit3=audit3.join(lag3,on=KEY,validate='one_to_one')
print('Lag4 availability:',audit3.lag4_ok.fillna(False).mean())
print('Lag4 raw correlation:',audit3[['turn_angle_rad','lag4_turn_rad']].corr().iloc[0,1])
summary3.to_csv(V3_DIR/'pre_model_audit.csv')
# Preserve coordinate lookup for follow-up measurement checks.
geom3=wr_motion.set_index(KEY)[['x','y']]
print('2025 not inspected for model choice.')


In [ ]:
# 5B. Frozen V3 specification + 2023 -> 2024 development validation.
# No 2025 rows or 2024 outcomes enter fitted preprocessing.
from scipy.optimize import minimize
from scipy.special import expit, logit, i0e as sp_i0e, i1e as sp_i1e
from scipy.stats import vonmises
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge
import joblib
V3_CONFIG = dict(knots=4, degree=3, mean_prior_sd=.10, zero_prior_rate=.08,
                 validation_train_year=2023, validation_year=2024, bin_width_deg=1.,
                 seed=2030, draws=1000, tune=1000, chains=4, target_accept=.95)
(V3_DIR/'frozen_spec.json').write_text(json.dumps(V3_CONFIG,indent=2))
# Same 2790 rows as V1/V2. Every retained lag must be within its segment.
v3_data = audit3.copy().reset_index(drop=True)
assert v3_data.lag4_ok.all()
v3_data['lag4_missing']=v3_data.lag4_turn_rad.isna().astype(float)
print('Missing valid lag4 angles:',int(v3_data.lag4_missing.sum()),'; zero imputation + explicit missing indicator')
v3_data['lag4_turn_rad']=v3_data.lag4_turn_rad.fillna(0.)
assert not v3_data.draft_year.eq(2025).any()

def prepare_v3_design(fit_df, apply_df, dynamic):
    cols=['speed_in_yd_s','signed_accel_yd_s2']
    c=fit_df[cols].mean(); s=fit_df[cols].std().replace(0,1)
    spline=SplineTransformer(n_knots=4,degree=3,include_bias=False,extrapolation='constant')
    spline.fit(fit_df[['progress']])
    sq_fit=((fit_df.speed_in_yd_s-c.iloc[0])/s.iloc[0])**2
    sq_c,sq_s=float(sq_fit.mean()),float(sq_fit.std())
    lag_abs_c=float(fit_df.lag4_turn_rad.abs().mean()); lag_abs_s=float(fit_df.lag4_turn_rad.abs().std())
    def raw(d):
        z=(d[cols]-c)/s
        dummies=np.column_stack([d.drill_name.eq(k).astype(float) for k in DRILLS])
        basis=spline.transform(d[['progress']])
        mean=np.column_stack([np.ones(len(d)),z.to_numpy(),d.prev_turn_rad,dummies[:,1:],
                               *[basis*dummies[:,j,None] for j in range(len(DRILLS))]])
        if dynamic: mean=np.column_stack([mean,d.lag4_turn_rad,d.lag4_missing])
        kap=np.column_stack([np.ones(len(d)),z.speed_in_yd_s,
                            (z.speed_in_yd_s**2-sq_c)/sq_s,z.signed_accel_yd_s2,dummies[:,1:]])
        if dynamic:
            kap=np.column_stack([kap,(d.lag4_turn_rad.abs()-lag_abs_c)/lag_abs_s])
        zero=np.column_stack([np.ones(len(d)),z.speed_in_yd_s,d.progress-.5,dummies[:,1:],
                              (d.lag4_turn_rad.eq(0)&d.lag4_missing.eq(0)).astype(float),d.lag4_missing])
        return mean.astype(float),kap.astype(float),zero.astype(float)
    mf,kf,zf=raw(fit_df); ma,ka,za=raw(apply_df)
    Q,R=np.linalg.qr(mf,mode='reduced'); Rscaled=R/np.sqrt(len(fit_df))
    assert np.linalg.matrix_rank(mf)==mf.shape[1]
    return (Q*np.sqrt(len(fit_df)),kf,zf),(np.linalg.solve(Rscaled.T,ma.T).T,ka,za),dict(
        spline=spline,centers=c,scales=s,speed_sq_center=sq_c,speed_sq_scale=sq_s,
        lag_abs_center=lag_abs_c,lag_abs_scale=lag_abs_s,R_scaled=Rscaled,dynamic=dynamic)

def fit_population_map(d,design,hurdle):
    X,K,H=design; y=d.turn_angle_rad.to_numpy(); iz=y==0; keep=~iz if hurdle else np.ones(len(y),bool)
    a,b,c=X.shape[1],K.shape[1],H.shape[1]
    kmean=np.zeros(b); kmean[0]=np.log(400.)
    ksd=np.full(b,.7); ksd[0]=1.5
    amean=np.zeros(c); amean[0]=logit(.08)
    asd=np.ones(c); asd[0]=1.5
    ini=Ridge(alpha=2,fit_intercept=False).fit(X[keep],np.tan(y[keep]/2)).coef_
    x0=np.r_[ini,kmean,amean] if hurdle else np.r_[ini,kmean]
    def fun(v):
        beta=v[:a]; gamma=v[a:a+b]
        eta=X@beta; mu=2*np.arctan(eta); kap=np.exp(K@gamma); delta=y-mu
        ll=kap*(np.cos(delta)-1)-np.log(2*np.pi)-np.log(sp_i0e(kap))
        loss=-ll[keep].sum()+.5*np.sum((beta/.10)**2)+.5*np.sum(((gamma-kmean)/ksd)**2)
        gm=kap*np.sin(delta)*2/(1+eta**2); gm[~keep]=0
        gk=kap*(np.cos(delta)-sp_i1e(kap)/sp_i0e(kap));gk[~keep]=0
        grad=np.r_[-X.T@gm+beta/.10**2,-K.T@gk+(gamma-kmean)/ksd**2]
        if hurdle:
            alpha=v[a+b:];eta0=H@alpha;p=expit(eta0)
            loss+=np.sum(np.logaddexp(0,eta0)-iz*eta0)+.5*np.sum(((alpha-amean)/asd)**2)
            grad=np.r_[grad,H.T@(p-iz)+(alpha-amean)/asd**2]
        return float(loss),grad
    bounds=[(None,None)]*a+[(-2,12)]+[(-5,5)]*(b-1)+([(None,None)]*c if hurdle else [])
    par_scale=np.r_[np.full(a,.10),ksd,asd] if hurdle else np.r_[np.full(a,.10),ksd]
    def standardized_fun(w):
        value,gradient=fun(w*par_scale)
        return value,gradient*par_scale
    standardized_bounds=[(None if low is None else low/scale,None if high is None else high/scale) for (low,high),scale in zip(bounds,par_scale)]
    result=minimize(standardized_fun,x0/par_scale,jac=True,method='L-BFGS-B',bounds=standardized_bounds,
                    options={'maxiter':5000,'ftol':1e-9,'gtol':1e-5})
    result.x=result.x*par_scale
    if not result.success: raise RuntimeError(result.message)
    return result

def population_predict(v,design,hurdle):
    X,K,H=design;a=X.shape[1];b=K.shape[1]
    return 2*np.arctan(X@v[:a]),np.exp(K@v[a:a+b]),expit(H@v[a+b:]) if hurdle else np.zeros(len(X))

def common_bin_score(d,mu,kap,pzero):
    # Same observation space for continuous baseline and point-mass hurdle model.
    y=d.turn_angle_rad.to_numpy();w=np.radians(1.)
    centers_bin=np.floor(y/w+.5)*w;lo=centers_bin-w/2;hi=centers_bin+w/2
    continuous=vonmises.cdf(hi,kap,loc=mu)-vonmises.cdf(lo,kap,loc=mu)
    prob=(1-pzero)*np.maximum(continuous,0)+pzero*((lo<=0)&(hi>0))
    return -np.log(np.maximum(prob,1e-15))

fit_dev=v3_data.loc[v3_data.draft_year.eq(2023)].copy()
val_dev=v3_data.loc[v3_data.draft_year.eq(2024)].copy()
print('Development split:',len(fit_dev),'train frames /',len(val_dev),'validation frames;',val_dev.nfl_id.nunique(),'new players')
assert set(fit_dev.nfl_id).isdisjoint(val_dev.nfl_id)
dev_fits={};dev_scores=[];dev_by_player={}
for label,dyn in [('population_spline_baseline',False),('V3_dynamic_hurdle',True)]:
    df,da,trans=prepare_v3_design(fit_dev,val_dev,dyn)
    result=fit_population_map(fit_dev,df,dyn)
    mu,k,pz=population_predict(result.x,da,dyn)
    nll=common_bin_score(val_dev,mu,k,pz)
    predicted_angle=np.angle(pz+(1-pz)*np.exp(1j*mu)*(sp_i1e(k)/sp_i0e(k)))
    errors=np.angle(np.exp(1j*(val_dev.turn_angle_rad.to_numpy()-predicted_angle)))
    per=pd.DataFrame({'nfl_id':val_dev.nfl_id.to_numpy(),'bin_nll':nll}).groupby('nfl_id').mean()
    dev_by_player[label]=per.bin_nll
    dev_fits[label]=dict(optimizer=result,transform=trans,mu=mu,kappa=k,pzero=pz,nll=nll)
    dev_scores.append(dict(model=label,frame_mean_bin_NLL=float(nll.mean()),player_equal_bin_NLL=float(per.bin_nll.mean()),
                           angular_MAE_deg=float(np.degrees(np.abs(errors)).mean()),
                           zero_Brier=float(np.mean((pz-val_dev.turn_angle_rad.eq(0).to_numpy())**2))))
dev_results=pd.DataFrame(dev_scores);display(dev_results)
delta=dev_by_player['V3_dynamic_hurdle']-dev_by_player['population_spline_baseline']
boot_rng=np.random.default_rng(2030)
bootstrap=np.array([boot_rng.choice(delta.to_numpy(),len(delta),replace=True).mean() for _ in range(2000)])
dev_comparison=dict(player_equal_delta_V3_minus_baseline=float(delta.mean()),
                    bootstrap_ci95_low=float(np.quantile(bootstrap,.025)),bootstrap_ci95_high=float(np.quantile(bootstrap,.975)),
                    split='2023->2024 development, previously used for V1/V2 training; not external test',
                    comparison='regularized population MAP models; not a refit of hierarchical V2')
print(dev_comparison)
print('Negative delta = improved common 1-degree-bin predictive log score. No 2025 tuning.')
dev_results.to_csv(V3_DIR/'development_validation.csv',index=False)
(V3_DIR/'development_comparison.json').write_text(json.dumps(dev_comparison,indent=2))
joblib.dump(dev_fits,V3_DIR/'development_models.joblib')


In [ ]:
# 5C. V3 full Bayesian fit. Existing V1/V2 posteriors remain unchanged.
# The hurdle component describes recorded exact zeros; it is not proof of a physical state.
if dev_comparison['player_equal_delta_V3_minus_baseline'] >= 0:
    raise RuntimeError('V3 did not improve the development log score; review before full fit.')
design3,_,transform3=prepare_v3_design(v3_data,v3_data,True)
Xm3,Xk3,Xz3=design3
init_map3=fit_population_map(v3_data,design3,True)
y3=v3_data.turn_angle_rad.to_numpy(); zero3=y3==0; nonzero3=np.flatnonzero(~zero3)
a3,b3,c3=Xm3.shape[1],Xk3.shape[1],Xz3.shape[1]
# Stable index ordering exactly matches the current V3 rows.
player3_ids=np.sort(v3_data.nfl_id.unique())
player3_idx=pd.Categorical(v3_data.nfl_id,categories=player3_ids).codes
attempt3_labels=v3_data[SEQ].astype(str).agg('|'.join,axis=1)
attempt3_idx,attempt3_levels=pd.factorize(attempt3_labels,sort=True)
coords3={'mean_feature':np.arange(a3),'kappa_feature':np.arange(b3),
         'zero_feature':np.arange(c3),'player':player3_ids.astype(str),
         'attempt':attempt3_levels.tolist()}
with pm.Model(coords=coords3) as circular_model_v3:
    beta3=pm.Normal('beta_mu',0,.10,dims='mean_feature')
    kg3=np.zeros(b3);kg3[0]=np.log(400.);ks3=np.full(b3,.7);ks3[0]=1.5
    gamma3=pm.Normal('beta_kappa',kg3,ks3,dims='kappa_feature')
    azmean3=np.zeros(c3);azmean3[0]=logit(.08);azsd3=np.ones(c3);azsd3[0]=1.5
    alpha3=pm.Normal('beta_zero',azmean3,azsd3,dims='zero_feature')
    sigma_p3=pm.HalfNormal('sigma_player',.7)
    pz3=pm.Normal('player_z',0,1,dims='player')
    effect_p3=pm.Deterministic('player_effect',sigma_p3*pz3,dims='player')
    sigma_a3=pm.HalfNormal('sigma_attempt_mean',.03)
    za3=pm.Normal('attempt_z',0,1,dims='attempt')
    ame3=sigma_a3*(za3[attempt3_idx]-pt.mean(za3))
    mu3=2*pt.arctan(pt.dot(Xm3,beta3)+ame3)
    kap3=pt.exp(pt.dot(Xk3,gamma3)+effect_p3[player3_idx])
    pm.Bernoulli('recorded_zero',logit_p=pt.dot(Xz3,alpha3),observed=zero3.astype('int8'))
    pm.CustomDist('nonzero_turn',mu3[nonzero3],kap3[nonzero3],
                  logp=stable_vonmises_logp,observed=y3[nonzero3])
    initial3={'beta_mu':init_map3.x[:a3],'beta_kappa':init_map3.x[a3:a3+b3],
              'beta_zero':init_map3.x[a3+b3:]}
    assert np.isfinite(circular_model_v3.compile_logp()(circular_model_v3.initial_point()))
    idata_v3=pm.sample(draws=1000,tune=1000,chains=4,cores=2,target_accept=.95,
                       init='adapt_diag',initvals=initial3,random_seed=2030,
                       idata_kwargs={'log_likelihood':False})
idata_v3.to_netcdf(V3_DIR/'posterior.nc')
joblib.dump(transform3,V3_DIR/'design_transform.joblib')
v3_data.to_csv(V3_DIR/'model_data_2023_2024.csv',index=False)
np.savez_compressed(V3_DIR/'designs.npz',mean=Xm3,kappa=Xk3,zero=Xz3,player_index=player3_idx,attempt_index=attempt3_idx)
print('V3 posterior saved:',V3_DIR)
print('2025 not used. V3 is an exploratory recorded-angle hurdle model, not a physical mechanism or NFL ranking.')


In [ ]:
# 5D. V3 diagnostics, same-row V2 comparison, and full saved outputs.
vars3=['beta_mu','beta_kappa','beta_zero','sigma_player','player_z','sigma_attempt_mean','attempt_z']
diag3=az.summary(idata_v3,var_names=vars3,hdi_prob=.95)
r3hat=az.rhat(idata_v3,var_names=vars3);e3ess=az.ess(idata_v3,var_names=vars3,method='bulk')
max_rhat3=max(float(v.max()) for v in r3hat.data_vars.values())
min_ess3=min(float(v.min()) for v in e3ess.data_vars.values())
div3=int(idata_v3.sample_stats.diverging.sum());bfmi3=float(np.min(az.bfmi(idata_v3)))
depth3=int(idata_v3.sample_stats.tree_depth.max())
sampler_ok3=bool(div3==0 and max_rhat3<=1.01 and min_ess3>=400 and bfmi3>.3 and depth3<10)
diag3.to_csv(V3_DIR/'mcmc_diagnostics.csv')
p3post=idata_v3.posterior.stack(sample=('chain','draw'))
s3rng=np.random.default_rng(2031);sel3=s3rng.choice(p3post.sizes['sample'],800,replace=False)
bm3=p3post.beta_mu.transpose('sample','mean_feature').values[sel3]
bk3=p3post.beta_kappa.transpose('sample','kappa_feature').values[sel3]
ba3=p3post.beta_zero.transpose('sample','zero_feature').values[sel3]
u3=p3post.player_effect.transpose('sample','player').values[sel3]
az3=p3post.attempt_z.transpose('sample','attempt').values[sel3]
sa3=p3post.sigma_attempt_mean.values[sel3]
pmu3=2*np.arctan(bm3@Xm3.T+sa3[:,None]*(az3[:,attempt3_idx]-az3.mean(axis=1,keepdims=True)))
pkap3=np.exp(bk3@Xk3.T+u3[:,player3_idx]);pzero3=expit(ba3@Xz3.T)
yrep3=s3rng.vonmises(pmu3,pkap3)
yrep3[s3rng.random(yrep3.shape)<pzero3]=0.

def summary_checks3(obs,rep):
    obs_stat=np.array([np.cos(obs).mean(),np.sin(obs).mean(),np.quantile(np.abs(obs),.9),
                       np.mean(obs==0),np.mean(np.abs(obs)<=np.radians(.5))])
    rep_stat=np.column_stack([np.cos(rep).mean(axis=1),np.sin(rep).mean(axis=1),
                               np.quantile(np.abs(rep),.9,axis=1),(rep==0).mean(axis=1),
                               (np.abs(rep)<=np.radians(.5)).mean(axis=1)])
    result=pd.DataFrame({'statistic':['mean_cos','mean_sin','abs_angle_90pct','exact_zero_rate','within_half_degree'],
                         'observed':obs_stat,'predictive_low':np.quantile(rep_stat,.025,axis=0),
                         'predictive_high':np.quantile(rep_stat,.975,axis=0)})
    result['inside_interval']=result.observed.between(result.predictive_low,result.predictive_high)
    return result
ppc3=summary_checks3(y3,yrep3);display(ppc3)
ppc3.to_csv(V3_DIR/'conditional_predictive_checks.csv',index=False)
ppc3_ok=bool(ppc3.inside_interval.all())
bydrill3=[]
for d in DRILLS:
    mask=v3_data.drill_name.eq(d).to_numpy()
    pcheck=summary_checks3(y3[mask],yrep3[:,mask]);pcheck['drill_name']=d;bydrill3.append(pcheck)
bydrill3=pd.concat(bydrill3,ignore_index=True)
display(bydrill3.loc[bydrill3.statistic.isin(['mean_sin','exact_zero_rate'])])
bydrill3.to_csv(V3_DIR/'drill_predictive_checks.csv',index=False)
# Mixture circular expectation includes both the point mass and continuous component.
expected_complex3=pzero3+(1-pzero3)*(sp_i1e(pkap3)/sp_i0e(pkap3))*np.exp(1j*pmu3)
predangle3=np.angle(expected_complex3.mean(axis=0))
resid3=np.angle(np.exp(1j*(y3-predangle3)))
# Compare only adjacent sampled moments separated by 0.4s; never across a gap/segment.
def lag_residual3(values):
    t=v3_data[SEG+['timestamp']].copy();t['residual']=values
    t=t.sort_values(SEG+['timestamp']);gr=t.groupby(SEG)
    t['lag']=gr.residual.shift(1);t['lag_t']=gr.timestamp.shift(1)
    valid=(t.timestamp-t.lag_t).dt.total_seconds().sub(.4).abs().le(.025)
    return float(t.loc[valid,['residual','lag']].corr().iloc[0,1]),int(valid.sum())
corr_v3,pairs3=lag_residual3(resid3);corr_v2_same,_=lag_residual3(resid2)
# V2 on same five checks (zero-event calibration is an added test).
ppc2_extended=summary_checks3(y3,yrep2)
comparison3=pd.DataFrame([
 dict(version='V2',max_Rhat=max_rhat2,min_bulk_ESS=min_ess2,divergences=div2,sampler_pass=sampler_ok2,
      five_PPC_pass=bool(ppc2_extended.inside_interval.all()),gap_checked_residual_corr=corr_v2_same),
 dict(version='V3',max_Rhat=max_rhat3,min_bulk_ESS=min_ess3,divergences=div3,sampler_pass=sampler_ok3,
      five_PPC_pass=ppc3_ok,gap_checked_residual_corr=corr_v3)])
display(comparison3)
print('Same-row 0.4s residual pairs:',pairs3)
comparison3.to_csv(V3_DIR/'v2_v3_diagnostics.csv',index=False)
# Recorded zeros have no automatic biological interpretation.
status3=dict(sampler_checks_pass=sampler_ok3,conditional_PPC_pass=ppc3_ok,
             residual_corr=corr_v3,previous_residual_corr_same_pairs=corr_v2_same,
             development_comparison=dev_comparison,
             heldout_2025_used=False,NFL_validation_done=False,
             status='EXPLORATORY_NOT_READY_FOR_PLAYER_RANKING')
(V3_DIR/'status.json').write_text(json.dumps(status3,indent=2))
fig3,ax3=plt.subplots(2,2,figsize=(13,8))
for ax,rep,label in [(ax3[0,0],yrep2,'V2 conditional fit'),(ax3[0,1],yrep3,'V3 dynamic hurdle')]:
    bins3=np.linspace(-30,30,81)
    ax.hist(np.degrees(y3),bins=bins3,density=True,alpha=.4,label='Observed')
    ax.hist(np.degrees(rep).ravel(),bins=bins3,density=True,histtype='step',color='darkorange',label='Predictive')
    ax.set(title=label,xlabel='Turn angle (degrees)');ax.legend()
labels3=['Observed','V2 predictive','V3 predictive']
ax3[1,0].bar(labels3,[np.mean(y3==0),np.mean(yrep2==0),np.mean(yrep3==0)],color=['steelblue','gray','teal'])
ax3[1,0].set(title='Exact recorded-zero rate',ylabel='Proportion')
ax3[1,1].bar(['V2','V3'],[corr_v2_same,corr_v3],color=['gray','teal'])
ax3[1,1].axhline(0,color='black',linewidth=.7)
ax3[1,1].set(title='Residual correlation at 0.4s',ylabel='Pearson correlation')
plt.tight_layout();fig3.savefig(V3_DIR/'v3_diagnostics.png',dpi=160,bbox_inches='tight');plt.show()
print('V3 status:',status3['status'])
print('Conditional training fit only; development population models used 2023->2024; final V3 not tested on 2025.')


In [ ]:
# 5E. Durable result package and human-readable summary.
# This does not approve a player ranking or use 2025.
import shutil, base64
from IPython.display import HTML,display
notes3=(
    'V3 dynamic recorded-zero hurdle model\n'
    'Data: 69 WR players / 2790 frames, 2023-2024. 2025 untouched.\n'
    'Observed exact recorded zeros: 230 / 2790. A hurdle models recording distribution, not a proven physical state.\n'
    'Mean: drill-specific cubic progress splines, speed, signed acceleration, previous angle, lag-four angle, lag missing indicator, centered attempt random effect.\n'
    'Concentration: speed, speed squared, signed acceleration, drill, absolute lag-four angle, player random effect.\n'
    'Zero probability: speed, progress, drill, previous exact-zero indicator, lag missing indicator.\n'
    '198 lag-four angles failed prior angle QC; zero imputed with explicit missing indicators. No rows removed.\n'
    '2023->2024 validation compares regularized population MAP models, not the stored hierarchical V2.\n'
    f"Development equal-player log-score delta V3-baseline: {dev_comparison['player_equal_delta_V3_minus_baseline']:.6f}; bootstrap CI [{dev_comparison['bootstrap_ci95_low']:.6f}, {dev_comparison['bootstrap_ci95_high']:.6f}].\n"
    'This is development evidence: 2024 was previously used in V1/V2 training, not a pristine external test.\n'
    f'Full V3 max Rhat: {max_rhat3:.6f}; min bulk ESS: {min_ess3:.1f}; divergences: {div3}.\n'
    f'Conditional five-summary PPC all passed: {ppc3_ok}.\n'
    f'Residual lag correlation on same valid 0.4s pairs: V2 {corr_v2_same:.6f}; V3 {corr_v3:.6f}.\n'
    'PPC conditions on observed lagged angles and fitted training effects; no full trajectory simulation or new-cohort Bayesian validation.\n'
    'No validated NFL performance claims or player ranking.\n'
)
(V3_DIR/'README_results.txt').write_text(notes3)
# Save combined files in a separate versioned archive.
archive3=Path(shutil.make_archive(str(DATA_DIR/'wr_bayesian_v3_results'),'zip',root_dir=V3_DIR))
print(notes3)
print('V3 result ZIP:',archive3,'MiB:',round(archive3.stat().st_size/1024**2,2))
# Visible UI download links, not an external transmission.
for pth,title in [(V3_DIR/'v3_diagnostics.png','V3 diagnostics figure'),(V3_DIR/'status.json','V3 status JSON')]:
    mime='image/png' if pth.suffix=='.png' else ('application/json' if pth.suffix=='.json' else 'application/zip')
    payload3=base64.b64encode(pth.read_bytes()).decode('ascii')
    if pth.suffix=='.png':display(HTML(f'<img alt="V3 diagnostics export" src="data:{mime};base64,{payload3}" style="max-width:100%">'))
    else:display(HTML(f'<a download="{pth.name}" href="data:{mime};base64,{payload3}">{title}</a>'))


# V3 загварын үр дүн

2026-11-01. V1/V2-ийн үр дүнг хадгалж, V3-ийг Colab дээр сургаж дуусгав.

## Юуг өөрчилсөн бэ?

- Бичлэг дэх яг 0° өнцөг үүсэх магадлалыг тусдаа logistic хэсгээр тооцсон. Энэ нь бодит хөдөлгөөний тусдаа төлөв батлагдсан гэсэн үг биш.
- 0.4 секундын өмнөх өнцгийг нэмж, өмнөх хөдөлгөөний мэдээллийг ашигласан.
- 198 мөрийн өмнөх өнцөг QC шалгалтаар хүчинтэй бус байсан. Тэдгээрийг 0 утгаар нөхөж, дутуу гэдгийг тусдаа индикатороор тэмдэглэсэн. 2,790 мөрийг хэвээр хадгалсан.
- Сорил тус бүрийн замын явцыг cubic spline-ээр, оролдлогын дундаж чиглэл болон тоглогчийн өнцгийн хэлбэлзлийг random effect-ээр тооцсон.
- 2025 оныг сургалт болон тохируулгад ашиглаагүй.

## Сайжирсан зүйл

2023 оны 1,528 мөрөөр сургаж, 2024 оны 31 шинэ тоглогчийн 1,262 мөр дээр хөгжүүлэлтийн харьцуулалт хийсэн.

| Үзүүлэлт | Хялбаршуулсан population суурь загвар | V3 population хувилбар |
|---|---:|---:|
| Дундаж өнцгийн зөрүү | 3.412° | 3.251° |
| Тоглогч бүрд ижил жин өгсөн 1° bin-ийн negative log score, бага нь сайн | 2.67765 | 2.53413 |

Log score-ийн V3 - суурь ялгаа -0.14352, тоглогчоор bootstrap хийсэн 95% интервал [-0.17233, -0.11722]. Энэ нь хадгалсан hierarchical V2-той хийсэн шууд харьцуулалт биш. Хоёр population загварыг ижил нөхцөлд дахин сургасан. 2024 оныг өмнө нь V1/V2-д хэрэглэсэн тул энэ нь хөгжүүлэлтийн шалгалт, хараат бус эцсийн тест биш.

Бүрэн V3-ийг 2023-2024 оны 69 тоглогчийн 2,790 мөрөөр 4 chain, 1,000 tuning + 1,000 posterior draw нөхцөлөөр сургаж дуусгасан.

| MCMC болон үлдэгдэл | V2 | V3 |
|---|---:|---:|
| Maximum R-hat | 1.006988 | 1.007122 |
| Minimum bulk ESS | 653.4 | 1,077.5 |
| Divergence | 0 | 0 |
| Сонгосон sampler шалгалт | Тэнцсэн | Тэнцсэн |
| Ижил 2,556 хосын, 0.4 сек зайтай үлдэгдлийн хамаарал | 0.356864 | 0.193448 |

Яг 0° өнцгийн бодит хувь 8.2437%. V3-ийн conditional predictive 95% хүрээ [6.8459%, 9.6425%] тул энэ үзүүлэлт хүрээнд багтсан. Хамаарал буурсан ч бүрэн арилсан гэж үзэхгүй.

## Шийдэгдээгүй зүйл

- Нийт чиглэлийн mean sin шалгалт тэнцээгүй: бодит 0.000735; predictive хүрээ [-0.012242, -0.003222]. Curl сорилд энэ хазайлт үлдсэн.
- ±0.5° доторх бодит хувь 21.8996%; predictive хүрээ [14.5878%, 17.6353%]. Яг тэгтэй хэсгийг зассан ч тэгт маш ойр тасралтгүй өнцгүүдийг дутуу сэргээж байна.
- Таван ерөнхий predictive шалгалтын 3 нь тэнцсэн; бүгд тэнцээгүй.
- Predictive шалгалт нь ажиглагдсан өмнөх өнцөг болон сургалтын random effect-д нөхцөлдсөн. Бүтэн замын simulation, шинэ тоглогчийн Bayesian predictive тест биш.
- NFL амжилттай холбоог баталгаажуулаагүй; тоглогчдын чансаа гаргахгүй.

## Дараагийн судалгаа

Curl-ийн эерэг эргэлтийн хэсэг болон тэг орчмын нарийн/өргөн хэлбэлзлийг ялгах mixture эсвэл хэмжилтийн нарийвчлалыг тооцсон ажиглалтын загварыг турших. Сонгосон хөгжүүлэлтийн протоколоор шалгасны дараа 2025 оныг үнэлэх. Зөвхөн MCMC эсвэл training PPC тэнцэхэд тулгуурлан хувилбар сонгохгүй.

## Файлууд

- `wr_combine_bayesian_v3.ipynb`: кодын нөөц; локал файлд сургалтын бүх output суулгаагүй.
- `wr_bayesian_v3.py`: V3-ийн таван code cell.
- `wr_bayesian_v3_results.zip`: posterior.nc, preprocessing transforms, design arrays, сургалтын мөрүүд, CSV шалгалтууд, график, тохиргоо. Локал архивт кодын нөөцийг мөн нэмсэн.
- `wr_bayesian_v3_diagnostics.png`: V2/V3-ийн график.

Colab: https://colab.research.google.com/drive/17AvE9vNm-zXiGYLRKKnwX6sUCOwmuuGy#scrollTo=eNGFrAuoz_oo
